# Dana’s Bakery — deterministic LangGraph orders
Run all cells in order in Google Colab. Orders contain only `item`, `quantity`, `day`, and `city`. All decisions and customer messages use Python rules and templates; no model, API key, or AI service is used.

Each invocation evaluates the same fixed inventory independently. Rugelach quantities count boxes; sourdough quantities count loaves. The graph never splits or partially fulfills an order.


In [ ]:
%pip install -q "langgraph>=1.0,<2.0"


## State and fixed business data
`total=False` lets callers omit internal fields and lets validation handle missing inputs. Runtime validation rejects invalid product, quantity, and day values, including boolean quantities. Days use the full English weekday names shown below. City is a string input; empty means pickup. Internal fields are reset for each order, with pickup and rejected as safe defaults until processing succeeds.


In [ ]:
from typing import TypedDict
from math import isclose
from pprint import pprint
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class OrderState(TypedDict, total=False):
    item: str
    quantity: int
    day: str
    city: str
    original_item: str
    suggestion_count: int
    delivery_changed: bool
    subtotal: float
    discount: float
    status: str
    delivery: str
    total: float
    message: str
    steps: list[str]

PRICES = {"challah": 18, "babka": 45, "rugelach": 30, "sourdough": 25}
INVENTORY = {"challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4}
CHAIN = ("babka", "rugelach", "sourdough", "challah")
DAYS = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")
OUTPUT_FIELDS = {"status", "item", "delivery", "total", "message", "steps"}
assert len(OrderState.__annotations__) == 14

# Returns a new list, never mutating a previous State's chronological log.
def logged(state, *entries):
    return [*state["steps"], *entries]


## Validation, stock checks, and bounded substitutions
Validation records structured findings in `steps`; rejection reasons need no extra State field. Routing reuses `check_stock` for the original product and each alternative. Each selection advances one position in the chain and increments the counter, so at most two alternatives are checked.


In [ ]:
def validate_order(state: OrderState):
    item = state.get("item", "")
    quantity = state.get("quantity", 0)
    day = state.get("day", "")
    city = state.get("city", "")
    steps = ["VALIDATE:START"]
    if not isinstance(item, str) or item not in PRICES:
        steps.append("VALIDATE:INVALID_ITEM")
    if type(quantity) is not int or quantity <= 0:
        steps.append("VALIDATE:INVALID_QUANTITY")
    if not isinstance(day, str) or day not in DAYS:
        steps.append("VALIDATE:INVALID_DAY")
    if len(steps) == 1:
        steps.append("VALIDATE:OK")
    return dict(item=item, quantity=quantity, day=day, city=city,
                original_item=item, suggestion_count=0, delivery_changed=False,
                subtotal=0.0, discount=0.0, status="rejected", delivery="pickup",
                total=0.0, message="", steps=steps)


def check_stock(state: OrderState):
    item, quantity = state["item"], state["quantity"]
    available = INVENTORY[item]
    return {"steps": logged(state,
        f"STOCK:{item}:requested={quantity}:available={available}",
        "STOCK:SUFFICIENT" if available >= quantity else "STOCK:INSUFFICIENT")}


def find_alternative(state: OrderState):
    assert state["suggestion_count"] < 2
    index = CHAIN.index(state["item"]) + 1
    assert index < len(CHAIN)
    item = CHAIN[index]
    count = state["suggestion_count"] + 1
    return {"item": item, "suggestion_count": count,
            "steps": logged(state, f"ALTERNATIVE:{count}:{item}")}


## Pricing, delivery, messages, and finalization
The 15% discount applies only to a subtotal strictly above 150 NIS on a day other than Friday. The delivery fee is local to `check_delivery`. That node marks successfully processed orders accepted before message generation. Rejection bypasses pricing and delivery, records a reason in the log, and retains the last evaluated item. Finalization verifies and preserves status.


In [ ]:
def calculate_price(state: OrderState):
    subtotal = float(PRICES[state["item"]] * state["quantity"])
    eligible = subtotal > 150 and state["day"] != "Friday"
    discount = subtotal * 0.15 if eligible else 0.0
    reason = "APPLIED_15_PERCENT" if eligible else (
        "NONE_FRIDAY" if state["day"] == "Friday" else "NONE_SUBTOTAL_AT_MOST_150")
    return {"subtotal": subtotal, "discount": discount,
            "steps": logged(state, f"PRICE:{state['item']}:subtotal={subtotal:.2f}",
                            f"DISCOUNT:{reason}:amount={discount:.2f}")}


def check_delivery(state: OrderState):
    city = state["city"]
    delivery = "delivery" if city == "Jerusalem" else "pickup"
    changed = bool(city) and city != "Jerusalem"
    delivery_fee = 20.0 if delivery == "delivery" else 0.0
    total = state["subtotal"] - state["discount"] + delivery_fee
    return {"delivery": delivery, "delivery_changed": changed, "total": total,
            "status": "accepted",
            "steps": logged(state,
                f"DELIVERY:{delivery}:fee={delivery_fee:.2f}:changed={changed}",
                "STATUS:accepted")}


def reject_order(state: OrderState):
    invalid = next((entry.split(":", 1)[1] for entry in state["steps"]
                    if entry.startswith("VALIDATE:INVALID_")), None)
    if invalid:
        reason = invalid
    elif state["suggestion_count"] >= 2:
        reason = "INSUFFICIENT_STOCK"
    else:
        assert CHAIN.index(state["item"]) == len(CHAIN) - 1
        reason = "NO_ALTERNATIVE"
    return {"status": "rejected", "total": 0.0,
            "steps": logged(state, f"REJECT:{reason}")}


def generate_message(state: OrderState):
    if state["status"] == "rejected":
        reasons = {
            "INVALID_ITEM": "that product is not on our active menu",
            "INVALID_QUANTITY": "the quantity must be a positive whole number (not a boolean)",
            "INVALID_DAY": "please provide a valid full English weekday name",
            "INSUFFICIENT_STOCK": "we cannot fill the entire quantity after checking two alternatives",
            "NO_ALTERNATIVE": "we do not have enough stock and there are no further alternatives",
        }
        reason = next(entry.split(":", 1)[1] for entry in reversed(state["steps"])
                      if entry.startswith("REJECT:"))
        message = f"We're sorry, we cannot accept your order: {reasons[reason]}. No charge will be made."
    else:
        assert state["status"] == "accepted"
        units = {"challah": "challah(s)", "babka": "babka(s)",
                 "rugelach": "box(es) of rugelach", "sourdough": "sourdough loaf/loaves"}
        message = f"Thank you for ordering from Dana's Bakery! Confirmed: {state['quantity']} {units[state['item']]}. "
        if state["item"] != state["original_item"]:
            message += f"We substituted {state['original_item']} with {state['item']} to fulfill your full quantity. "
        if state["delivery_changed"]:
            message += "Delivery is available only in Jerusalem, so your order was automatically changed to pickup. "
        elif state["delivery"] == "delivery":
            message += "Delivery to Jerusalem is included (20.00 NIS delivery fee). "
        else:
            message += "Your order is for pickup. "
        if state["discount"]:
            message += f"Your 15% baked-goods discount is {state['discount']:.2f} NIS. "
        message += f"Final price: {state['total']:.2f} NIS. We look forward to seeing you!"
    return {"message": message, "steps": logged(state, f"MESSAGE:{state['status']}")}


def finalize_order(state: OrderState):
    assert OUTPUT_FIELDS <= state.keys()
    assert state["status"] in ("accepted", "rejected")
    assert state["delivery"] in ("pickup", "delivery")
    assert state["message"]
    if state["status"] == "accepted":
        assert "STATUS:accepted" in state["steps"]
        assert not any(s.startswith("REJECT:") for s in state["steps"])
        assert INVENTORY[state["item"]] >= state["quantity"]
        fee = 20.0 if state["delivery"] == "delivery" else 0.0
        assert isclose(state["total"], state["subtotal"] - state["discount"] + fee)
    else:
        assert any(s.startswith("REJECT:") for s in state["steps"])
    return {"status": state["status"],
            "total": 0.0 if state["status"] == "rejected" else state["total"],
            "steps": logged(state, f"FINALIZE:{state['status']}")}


## Conditional routing and graph compilation
Exactly eight Nodes are registered. Rejected and accepted paths both finish with message generation and finalization.


In [ ]:
def route_validation(state: OrderState):
    return "check_stock" if "VALIDATE:OK" in state["steps"] else "reject_order"


def route_stock(state: OrderState):
    if INVENTORY[state["item"]] >= state["quantity"]:
        return "calculate_price"
    if state["suggestion_count"] < 2 and CHAIN.index(state["item"]) < len(CHAIN) - 1:
        return "find_alternative"
    return "reject_order"


builder = StateGraph(OrderState)
for node in (validate_order, check_stock, find_alternative, calculate_price,
             check_delivery, generate_message, finalize_order, reject_order):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", route_validation,
                             {name: name for name in ("check_stock", "reject_order")})
builder.add_conditional_edges("check_stock", route_stock,
                             {name: name for name in ("calculate_price", "find_alternative", "reject_order")})
builder.add_edge("find_alternative", "check_stock")
builder.add_edge("calculate_price", "check_delivery")
builder.add_edge("check_delivery", "generate_message")
builder.add_edge("reject_order", "generate_message")
builder.add_edge("generate_message", "finalize_order")
builder.add_edge("finalize_order", END)
graph = builder.compile()


## Graph visualization
`draw_mermaid_png()` uses the Mermaid rendering service for this diagram, not an AI service. Internet access is needed for rendering. If the renderer is unavailable, the Mermaid source is displayed and order processing/tests can still run.


In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"PNG rendering unavailable: {type(exc).__name__}: {exc}")
    print(graph.get_graph().draw_mermaid())


## Example order
Two babkas are fulfilled with two boxes of rugelach; Jerusalem delivery brings the total to 80 NIS.


In [ ]:
result = graph.invoke({"item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"})
pprint({key: result[key] for key in ("status", "item", "delivery", "total", "message", "steps")})


## All ten required scenarios
Assertions check status, actual product, fulfillment, total, discount, alternative count, messages, and log order. The stock-check trace proves that the loop stops as soon as stock is sufficient and never checks more than two alternatives. Each invocation uses only the four public input fields.


In [ ]:
# item, quantity, day, city, status, sold item, delivery, total, alternatives, discount, changed, stock trace
CASES = [
    ("challah", 2, "Thursday", "Jerusalem", "accepted", "challah", "delivery", 56, 0, 0, False, ["challah"]),
    ("babka", 2, "Thursday", "Jerusalem", "accepted", "rugelach", "delivery", 80, 1, 0, False, ["babka", "rugelach"]),
    ("rugelach", 5, "Thursday", "", "accepted", "challah", "pickup", 90, 2, 0, False, ["rugelach", "sourdough", "challah"]),
    ("babka", 5, "Thursday", "Jerusalem", "rejected", "sourdough", "pickup", 0, 2, 0, False, ["babka", "rugelach", "sourdough"]),
    ("croissant", 1, "Monday", "Jerusalem", "rejected", "croissant", "pickup", 0, 0, 0, False, []),
    ("challah", 0, "Monday", "Jerusalem", "rejected", "challah", "pickup", 0, 0, 0, False, []),
    ("challah", 9, "Thursday", "Jerusalem", "accepted", "challah", "delivery", 157.70, 0, 24.30, False, ["challah"]),
    ("challah", 9, "Friday", "Jerusalem", "accepted", "challah", "delivery", 182, 0, 0, False, ["challah"]),
    ("challah", 2, "Monday", "Tel Aviv", "accepted", "challah", "pickup", 36, 0, 0, True, ["challah"]),
    ("challah", 2, "Monday", "", "accepted", "challah", "pickup", 36, 0, 0, False, ["challah"]),
]

def assert_result(result):
    assert set(result) == set(OrderState.__annotations__)
    assert OUTPUT_FIELDS <= result.keys()
    assert result["status"] in ("accepted", "rejected")
    assert result["steps"][0] == "VALIDATE:START"
    assert result["steps"][-2:] == [f"MESSAGE:{result['status']}", f"FINALIZE:{result['status']}"]
    assert len([s for s in result["steps"] if s.startswith("ALTERNATIVE:")]) == result["suggestion_count"] <= 2
    assert "REJECT:" not in result["message"]
    if result["status"] == "rejected":
        assert result["total"] == 0 and result["delivery"] == "pickup"
        assert not any(s.startswith(("PRICE:", "DISCOUNT:", "DELIVERY:")) for s in result["steps"])
        assert "sorry" in result["message"]
    else:
        assert str(result["quantity"]) in result["message"]
        assert f"{result['total']:.2f} NIS" in result["message"]
        assert result["delivery"] in result["message"].lower()
        assert ("substituted" in result["message"]) == (result["item"] != result["original_item"])
        assert ("automatically changed to pickup" in result["message"]) == result["delivery_changed"]
        prefixes = [s.split(":", 1)[0] for s in result["steps"]]
        assert prefixes.index("PRICE") < prefixes.index("DISCOUNT") < prefixes.index("DELIVERY") < prefixes.index("MESSAGE")

for number, case in enumerate(CASES, 1):
    item, quantity, day, city, status, sold, delivery, total, alternatives, discount, changed, trace = case
    result = graph.invoke(dict(item=item, quantity=quantity, day=day, city=city))
    assert_result(result)
    assert (result["status"], result["item"], result["delivery"]) == (status, sold, delivery), number
    assert result["quantity"] == quantity and result["original_item"] == item
    assert isclose(result["total"], total, abs_tol=1e-9), number
    assert isclose(result["discount"], discount, abs_tol=1e-9), number
    assert result["suggestion_count"] == alternatives
    assert result["delivery_changed"] is changed
    checked = [s.split(":")[1] for s in result["steps"] if ":requested=" in s]
    assert checked == trace, (number, checked)
    print(f"Required scenario {number}: passed")


## Additional validation and branch coverage
These checks exercise no-next-alternative rejection, missing/invalid inputs, boolean and noninteger quantities, direct sourdough sales, and repeat-invocation isolation. The exactly-150 discount boundary is tested directly at the pricing Node because no accepted order with the fixed inventory can produce that subtotal (five rugelach boxes exceed stock). No cancellation functionality is defined or added.


In [ ]:
base = dict(item="challah", quantity=1, day="Monday", city="Jerusalem")
invalid_orders = [
    ({**base, "quantity": q}, "INVALID_QUANTITY")
    for q in (-1, True, False, 1.5, "2", None, [], {})
] + [
    ({**base, "item": item}, "INVALID_ITEM") for item in ("", None, [], {}, "croissant")
] + [
    ({**base, "day": day}, "INVALID_DAY") for day in ("", "Funday", None, [], 5)
]
for missing, reason in (("item", "INVALID_ITEM"), ("quantity", "INVALID_QUANTITY"), ("day", "INVALID_DAY")):
    invalid_orders.append(({k: v for k, v in base.items() if k != missing}, reason))
for order, reason in invalid_orders:
    result = graph.invoke(order)
    assert_result(result)
    assert result["status"] == "rejected" and f"REJECT:{reason}" in result["steps"]
    assert result["item"] == order.get("item", "")

for item, quantity, expected_count in (("challah", 11, 0), ("sourdough", 11, 1), ("rugelach", 11, 2)):
    result = graph.invoke({**base, "item": item, "quantity": quantity})
    assert_result(result)
    expected_reason = "INSUFFICIENT_STOCK" if expected_count == 2 else "NO_ALTERNATIVE"
    assert f"REJECT:{expected_reason}" in result["steps"]
    assert result["suggestion_count"] == expected_count
    assert result["item"] == "challah"

result = graph.invoke({**base, "item": "sourdough", "quantity": 4})
assert_result(result)
assert result["item"] == "sourdough" and result["total"] == 120
result = graph.invoke({**base, "item": "rugelach", "quantity": 4})
assert_result(result)
assert result["item"] == "sourdough" and result["suggestion_count"] == 1

boundary = calculate_price(dict(item="rugelach", quantity=5, day="Thursday", steps=[]))
assert boundary["subtotal"] == 150.0 and boundary["discount"] == 0.0
inventory_before = INVENTORY.copy()
order = dict(item="challah", quantity=9, day="Thursday", city="Jerusalem")
first, second = graph.invoke(order), graph.invoke(order)
assert first == second and INVENTORY == inventory_before
first["steps"].append("TEST_ONLY")
assert "TEST_ONLY" not in second["steps"] and "TEST_ONLY" not in graph.invoke(order)["steps"]
assert order == dict(item="challah", quantity=9, day="Thursday", city="Jerusalem")
print("All additional validation, branch, boundary, and isolation checks passed.")
